In [ ]:
import json
import pandas as pd
from pathlib import Path

# 1. Define the directories containing your JSON files
folder_paths = [
    "path/to/your/folder_1",
    "path/to/your/folder_2",
    "path/to/your/folder_3"
]

output_csv = "compiled_merged_dataset.csv"

# This dictionary will use the file name as a key to group matching files together
compiled_data = {}

# 2. Iterate through each folder
for index, folder_path_str in enumerate(folder_paths):
    folder = Path(folder_path_str)
    
    # Create a dynamic column name for the description based on the folder number
    # (e.g., 'description_folder_1', 'description_folder_2', etc.)
    desc_column_name = f"description_folder_{index + 1}"
    
    # Find all JSON files in the current folder
    for file_path in folder.rglob('*.json'):
        file_key = file_path.name  # We use the filename to link the 3 versions together
        
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                # If this is the first time we are seeing this file, save the common data
                if file_key not in compiled_data:
                    compiled_data[file_key] = {
                        "file_name": file_key, # Keeping the filename for reference
                        "question": data.get("question", ""),
                        "original_source": data.get("original_source", ""),
                        "data_group": data.get("data_group", ""),
                        "data_point": data.get("data_point", ""),
                        "reference_1": data.get("reference_1", ""),
                        "reference_2": data.get("reference_2", ""),
                        "references": json.dumps(data.get("references", []))
                    }
                
                # Now, add the specific description from this folder
                compiled_data[file_key][desc_column_name] = data.get("description", "")
                
        except json.JSONDecodeError:
            print(f"Warning: Could not parse JSON in {file_path}. Skipping.")
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

# 3. Convert the grouped dictionary values into a Pandas DataFrame
print(f"Successfully merged {len(compiled_data)} unique files. Converting to CSV...")
# compiled_data.values() gives us just the rows, dropping the file_key dictionary structure
df = pd.DataFrame(list(compiled_data.values()))

# 4. Save to CSV
df.to_csv(output_csv, index=False, encoding='utf-8')
print(f"Done! Merged dataset saved to {output_csv}")